In [1]:
from pathlib import Path
from copy import deepcopy

import matplotlib.pyplot as plt

from contact_plots import get_data_from_files, create_and_save_percentile_paper_plot, get_dist_time_prob_data

In [2]:
data_folder = Path(r'C:\Users\martijnsparnaa\surfdrive\PhD\Activity scheduler\Simulations\2021_12\transfer\results_20211215')

contact_data_list, infra_and_measure_name_2_data = get_data_from_files(data_folder, convert_2_minutes=True)


In [3]:
cut_off_distances = ['1.0', '1.5', '2.0']

data_names_base = (
    'customer_customer_weight_over_contacts',
    'customer_customer_weight_over_agents',
    'customer_customer_contacts_over_agents',
    'staff_customer_weight_over_contacts',
    'staff_customer_weight_over_agents',
    'staff_customer_contacts_over_agents',
    'customer_staff_weight_over_agents',
    'customer_staff_contacts_over_agents',
)

parts_2_data_name = {}
for cut_off_distance in cut_off_distances:
    data_name_dict = {}
    for data_name in data_names_base:
        parts = data_name.split('_')
        interaction_type = '_'.join(parts[:2])
        data_type = '_'.join(parts[2:])
        if interaction_type not in data_name_dict:
            data_name_dict[interaction_type] = {}
            
        data_name_dict[interaction_type][data_type] = f'{cut_off_distance}_{data_name}'
    
    parts_2_data_name[cut_off_distance] = data_name_dict
    
measures_and_labels = (
    ('nn_nts_bvd', 'Base'),
    ('n_nts_bvd', 'Neighbourhoods'),
    ('nn_ts_bvd', 'Time slots'),
    ('nn_nts_svd', 'Shorter visits'),
    ('nn_nts_lvd', 'Longer visits')
)

infra_names = [
    'base_case',
    'busy_corner',
    'long_restaurant']

infra_names_2_label = {
    'base_case': 'Base case',
    'busy_corner': 'Busy corner', 
    'long_restaurant': 'Long restaurant'}

dist_time_combs = (
    ('1.0', 1),
    ('1.5', 15),    
    ('2.0', 15),
    ('1.5', 10)    
)

dist_time_base_combs = ('1.0', 1)


In [4]:
divider_config = {
    'left':1.25,  
    'gap_hor':0,
    'right':0.1,
    'bottom':1.00,
    'gap_ver':0,
    'top':0.30,
    'hor_axes_locs':[0]*4 + [1]*4 + [2]*4,
    'ver_axes_locs':[,0]*3
}

measure_names = [measure_name for measure_name, _ in measures_and_labels]
labels = [label for _, label in measures_and_labels]

main_ax_config = {'grid':{'visible':True, 'axis':'y'}, 'y_label_config':{'fontsize':9},
                 'x_tick_labels':labels, 'x_tick_labels_config':{'fontsize':9, 'rotation':45, 'ha':'right'},
                 'y_tick_labels_config':{'labelsize':9}, 'title_config':{'fontsize':10}}

v_line_config = {'linewidth':1.5, 'color':'0.6', 'linestyle':'--', 'zorder':2}

base_plot_config = {'measure_names':measure_names, 'main_ax_config':main_ax_config,
                   'v_line_config':v_line_config, 'use_legend':False}

interaction_type_labels = {'customer_customer':'Customer - Customer', 
                           'customer_staff':'Customer - Staff', 
                           'staff_customer':'Staff - Customer'}

line_config_base = {'linewidth':1, 'zorder':3}

dist_time_line_config = {
    ('1.0', 1): {'color':'r', 'marker':'+'},
    ('1.5', 15): {'color':'b', 'marker':'x'},    
    ('2.0', 15): {'color':'g', 'marker':'o'},
    ('1.5', 10): {'color':'c', 'marker':'*'}    
}

fig_config_base = {'figsize':(6.5, 4)}

save_pd = Path(r'C:\Users\martijnsparnaa\surfdrive\PhD\Graphics')


In [5]:
%matplotlib wx

interaction_type_all = 'customer_customer'

save_filename = save_pd.joinpath(f'{interaction_type_all}_perc_line.pdf')

fig_config = deepcopy(fig_config_base)
fig_config.update({'title': f'{interaction_type_all}'})

plot_config = deepcopy(base_plot_config)
plots_config = []

for ii, infra_name in enumerate(infra_names):
    dist_time_prob_data = get_dist_time_prob_data(interaction_type_all, infra_name, measure_names, dist_time_combs, dist_time_base_combs, parts_2_data_name, infra_and_measure_name_2_data)
    plot_config_sub = deepcopy(plot_config)
    plot_config_sub['main_ax_config']['x_tick_labels'] = []
    lines = []
    for key, line_config_sub in dist_time_line_config.items():
        line_config = deepcopy(line_config_base)
        line_config.update(line_config_sub)
        lines.append({'y':dist_time_prob_data['prob'][key], 'line_config':line_config})
    plot_config_sub['lines'] = lines
    plot_config_sub['main_ax_config']['title'] = infra_names_2_label[infra_name]
    if ii == 0:
        plot_config_sub['main_ax_config']['y_label'] = 'Probability [-]'
    else:
        plot_config_sub['main_ax_config']['y_tick_labels'] = []
    plots_config.append(plot_config_sub)
    
    plot_config_sub = deepcopy(plot_config)
    if ii == 0:
        plot_config_sub['main_ax_config']['y_label'] = 'Total # contacts [-]'
    else:
        plot_config_sub['main_ax_config']['y_tick_labels'] = []

    plot_config_sub['main_ax_config']['x_tick_labels'] = []
    line_config = deepcopy(line_config_base)
    line_config.update({'marker':'x'})
    plot_config_sub['lines'] = [{'y':dist_time_prob_data['total_count'][dist_time_combs[0]], 'line_config':line_config}]
    plots_config.append(plot_config_sub)
    
    plot_config_sub = deepcopy(plot_config)
    if ii == 0:
        plot_config_sub['main_ax_config']['y_label'] = 'Total # close contacts [-]'
    else:
        plot_config_sub['main_ax_config']['y_tick_labels'] = []

    plot_config_sub['main_ax_config']['x_tick_labels'] = []
    lines = []
    for key, line_config_sub in dist_time_line_config.items():
        line_config = deepcopy(line_config_base)
        line_config.update(line_config_sub)
        lines.append({'y':dist_time_prob_data['count'][key], 'line_config':line_config})
    plot_config_sub['lines'] = lines
    plots_config.append(plot_config_sub)
    
    plot_config_sub = deepcopy(plot_config)
    if ii == 0:
        plot_config_sub['main_ax_config']['y_label'] = 'Overlap [-]'    
    else:
        plot_config_sub['main_ax_config']['y_tick_labels'] = []
    lines = []
    for key, line_config_sub in dist_time_line_config.items():
        if key == dist_time_base_combs:
            continue
        line_config = deepcopy(line_config_base)
        line_config.update(line_config_sub)
        lines.append({'y':overlap_y, 'line_config':line_config})
    plot_config_sub['lines'] = lines
    plots_config.append(plot_config_sub)

create_and_save_percentile_paper_plot(fig_config, save_filename, divider_config, plots_config)



ValueError: operands could not be broadcast together with shapes (63572,) (7618,) 

In [ ]:
interaction_type_all = 'staff_customer'

save_filename = save_pd.joinpath(f'{interaction_type_all}_perc_line.pdf')

fig_config = deepcopy(fig_config_base)
fig_config.update({'title': f'{interaction_type_all}'})

plot_config = deepcopy(base_plot_config)
plots_config = []

for ii, infra_name in enumerate(infra_names):
    dist_time_prob_data = get_dist_time_prob_data(interaction_type_all, infra_name, measure_names, dist_time_combs, dist_time_base_combs, parts_2_data_name, infra_and_measure_name_2_data)
    plot_config_sub = deepcopy(plot_config)
    plot_config_sub['main_ax_config']['x_tick_labels'] = []
    lines = []
    for key, line_config_sub in dist_time_line_config.items():
        line_config = deepcopy(line_config_base)
        line_config.update(line_config_sub)
        lines.append({'y':dist_time_prob_data['prob'][key], 'line_config':line_config})
    plot_config_sub['lines'] = lines
    plot_config_sub['main_ax_config']['title'] = infra_names_2_label[infra_name]
    if ii == 0:
        plot_config_sub['main_ax_config']['y_label'] = 'Probability [-]'
    else:
        plot_config_sub['main_ax_config']['y_tick_labels'] = []
    plots_config.append(plot_config_sub)
    
    plot_config_sub = deepcopy(plot_config)
    if ii == 0:
        plot_config_sub['main_ax_config']['y_label'] = 'Total # contacts [-]'
    else:
        plot_config_sub['main_ax_config']['y_tick_labels'] = []

    plot_config_sub['main_ax_config']['x_tick_labels'] = []
    line_config = deepcopy(line_config_base)
    line_config.update({'marker':'x'})
    plot_config_sub['lines'] = [{'y':dist_time_prob_data['total_count'][dist_time_combs[0]], 'line_config':line_config}]
    plots_config.append(plot_config_sub)
    
    plot_config_sub = deepcopy(plot_config)
    if ii == 0:
        plot_config_sub['main_ax_config']['y_label'] = 'Total # close contacts [-]'
    else:
        plot_config_sub['main_ax_config']['y_tick_labels'] = []

    plot_config_sub['main_ax_config']['x_tick_labels'] = []
    lines = []
    for key, line_config_sub in dist_time_line_config.items():
        line_config = deepcopy(line_config_base)
        line_config.update(line_config_sub)
        lines.append({'y':dist_time_prob_data['count'][key], 'line_config':line_config})
    plot_config_sub['lines'] = lines
    plots_config.append(plot_config_sub)
    
    plot_config_sub = deepcopy(plot_config)
    if ii == 0:
        plot_config_sub['main_ax_config']['y_label'] = 'Overlap [-]'    
    else:
        plot_config_sub['main_ax_config']['y_tick_labels'] = []
    lines = []
    for key, line_config_sub in dist_time_line_config.items():
        if key == dist_time_base_combs:
            continue
        line_config = deepcopy(line_config_base)
        line_config.update(line_config_sub)
        lines.append({'y':overlap_y, 'line_config':line_config})
    plot_config_sub['lines'] = lines
    plots_config.append(plot_config_sub)

create_and_save_percentile_paper_plot(fig_config, save_filename, divider_config, plots_config)

